In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten
from keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import confusion_matrix,classification_report
import matplotlib.pyplot as plt
import numpy as np
from keras.callbacks import ModelCheckpoint, EarlyStopping

In [3]:
train_data_dir = '/content/drive/My Drive/<PASTA_DO_PROJETO>/data/train'
validation_data_dir = '/content/drive/My Drive/<PASTA_DO_PROJETO>/data/test'
test_data_dir = '/content/drive/My Drive/<PASTA_DO_PROJETO>/data/val'

In [4]:
train_datagen = ImageDataGenerator(rescale=1./255)
validation_datagen = ImageDataGenerator(rescale=1./255)

In [5]:
batch_size = 16
img_rows, img_cols = 224, 224

In [11]:
train_generator = train_datagen.flow_from_directory(train_data_dir,
                                                    target_size=(img_rows, img_cols),
                                                    batch_size=batch_size,
                                                    class_mode='categorical')

validation_generator = validation_datagen.flow_from_directory(validation_data_dir,
                                                              target_size=(img_rows, img_cols),
                                                              batch_size=batch_size,
                                                              class_mode='categorical')

Found 83504 images belonging to 4 classes.
Found 968 images belonging to 4 classes.


In [7]:
img_shape = (img_rows, img_cols, 3)

In [8]:
mobile = tf.keras.applications.MobileNetV2(input_shape=img_shape,
                                              include_top=False,
                                              weights='imagenet')

9412608/9406464 [==============================] - 0s 0us/step


In [9]:
mobile.trainable = False

In [10]:
model = tf.keras.Sequential([
                          mobile,
                          keras.layers.GlobalAveragePooling2D(),
                          keras.layers.Dense(4, activation='softmax')])

In [12]:
print(model.summary())

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
mobilenetv2_1.00_224 (Functi (None, 7, 7, 1280)        2257984   
_________________________________________________________________
global_average_pooling2d (Gl (None, 1280)              0         
_________________________________________________________________
dense (Dense)                (None, 4)                 5124      
Total params: 2,263,108
Trainable params: 5,124
Non-trainable params: 2,257,984
_________________________________________________________________
None


In [17]:
checkpoint = ModelCheckpoint("base_model.h5",
                             monitor="loss",
                             mode="min",
                             save_best_only = True,
                             verbose=1)

earlystop = EarlyStopping(monitor = 'loss',
                          patience = 1,
                          verbose = 1,
                          restore_best_weights = True)

callbacks = [earlystop, checkpoint]

In [18]:
model.compile(loss='categorical_crossentropy',
             optimizer=Adam(learning_rate=0.001),
             metrics=['accuracy'])

In [19]:
epochs = 5

nb_train_samples = 5000
nb_val_samples = 1000

In [ ]:
history = model.fit(train_generator,
                    steps_per_epoch = nb_train_samples // batch_size,
                    epochs = epochs,
                    callbacks = callbacks,
                    validation_data = validation_generator,
                    validation_steps = nb_val_samples // batch_size)

Epoch 1/5
124/312 [==========>...................] - ETA: 15:37 - loss: 0.7909 - accuracy: 0.6935

In [49]:
#model.load_weights('base_model.h5')

/usr/local/lib/python3.7/dist-packages/tensorflow/python/keras/utils/generic_utils.py:497: CustomMaskWarning: Custom mask layers require a config and must override get_config. When loading, the custom mask layer must be passed to the custom_objects argument.
  category=CustomMaskWarning)


In [ ]:
valid_loss, valid_accuracy = model.evaluate_generator(generator = validation_generator, verbose = 1)

print('Validation Accuracy: ', round((valid_accuracy * 100), 2), "%")

In [ ]:
class_labels = validation_generator.class_indices
class_labels = {v: k for k, v in class_labels.items()}
classes = list(class_labels.values())

In [ ]:
y_pred = model.predict(validation_generator, nb_val_samples // batch_size+1, verbose=1)

In [ ]:
y_pred_label = np.argmax(y_pred, axis=1)

In [ ]:
val_generator.classes.shape,y_pred_label.shape

In [ ]:
#Confusion Matrix and Classification Report
print('Confusion Matrix')
print(confusion_matrix(validation_generator.classes, y_pred_label))
print('Classification Report')
print(classification_report(validation_generator.classes, y_pred_label, target_names=classes))

In [ ]:
plt.figure(figsize=(8,8))
cnf_matrix = confusion_matrix(validation_generator.classes, y_pred_label)

plt.imshow(cnf_matrix, interpolation='nearest')
plt.colorbar()
tick_marks = np.arange(len(classes))
_ = plt.xticks(tick_marks, classes, rotation=90)
_ = plt.yticks(tick_marks, classes)
plt.show();